In [ ]:
import os
import pandas as pd
import shutil
from tqdm import tqdm

# 获取当前工作目录
input_dir = os.path.join(os.getcwd(), 'charge2')  # 'charge2'文件夹路径
print(f"Processing files in: {input_dir}")

# 创建 "Unqualified_Clip" 文件夹（如果不存在）
unqualified_dir = os.path.join(input_dir, 'Unqualified_Clip')
if not os.path.exists(unqualified_dir):
    os.makedirs(unqualified_dir)

# 初始化一个 DataFrame 来保存最终结果
results_df = pd.DataFrame(columns=['文件名', 'SOH 值', '分母值', '累计里程', 'SOC起始值', 'SOC结束值', '单次容量', '电池温度', '环境温度'])

# 获取所有文件名，过滤出 `fixed_data*_charge*` 文件
all_files = [f for f in os.listdir(input_dir) if f.endswith('.xlsx') and f.startswith('fixed_data') and '_charge' in f]

# 设置总进度条
with tqdm(total=len(all_files), desc="Processing Files", unit="file") as pbar:
    # 遍历文件夹中的文件，按前缀 fixed_data1 到 fixed_data11
    for i in range(4, 16):  # 按照前缀 fixed_data4 到 fixed_data15
        file_pattern = f'fixed_data{i}_charge'
        for file_name in all_files:
            if file_name.startswith(file_pattern):
                file_path = os.path.join(input_dir, file_name)
                
                # 读取 Excel 文件
                df = pd.read_excel(file_path, engine='openpyxl')
                
                # 确保 DataFrame 至少有 17 列和足够的行用于计算
                if len(df.columns) < 17 or len(df) < 7:
                    print(f"跳过 {file_name}，因为列或行数不足。")
                    pbar.update(1)
                    continue
                
                # 计算 SOH 值
                try:
                    # 计算 num_value（第六列所有值的和的绝对值除以360）
                    num_value = abs(df.iloc[:, 5].sum()) / 360  # 计算第六列所有值的和的绝对值除以 360
                    
                    # 计算 den_value（分母值）
                    den_value = df.iloc[-5:, 6].mean() - df.iloc[1:6, 6].mean()
                    if den_value == 0 or den_value < 20:
                        raise ValueError("分母无效")
                    soh_value = (num_value * 100) / den_value
                except Exception as e:
                    print(f"{file_name} 不合格，原因：{str(e)}")
                    shutil.move(file_path, os.path.join(unqualified_dir, file_name))
                    pbar.update(1)
                    continue
                
                # 在 DataFrame 中添加 SOH 列
                df['SOH'] = soh_value
                
                # 保存更新后的 Excel 文件
                df.to_excel(file_path, index=False, engine='openpyxl')
                
                # 收集结果
                result_row = pd.DataFrame({
                    '文件名': [file_name],
                    'SOH 值': [soh_value],
                    '分母值': [den_value],
                    '累计里程': [df.iloc[-1, 3]],
                    'SOC起始值': [df.iloc[1:6, 6].mean()],
                    'SOC结束值': [df.iloc[-5:, 6].mean()],
                    '单次容量': [num_value],
                    '电池温度': [df.iloc[-1, 22]],
                    '环境温度': [df.iloc[1:6, 16].mean()]
                })

                # 使用 concat 替代 append
                results_df = pd.concat([results_df, result_row], ignore_index=True)
                
                # 更新进度条
                pbar.update(1)

# 保存结果到 SOH_Result.xlsx
results_file = os.path.join(input_dir, 'SOH_Result.xlsx')
results_df.to_excel(results_file, index=False, engine='openpyxl')

print("处理完成，结果已保存到 SOH_Result.xlsx。")
